In [ ]:
!pip install torch numpy transformers datasets rouge_score bert-score pandas matplotlib tqdm causal-conv1d mamba-ssm

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cu

In [ ]:
import time
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    MambaConfig,
    MambaForCausalLM
)
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

max_input_length = 1024
max_new_tokens = 512
batch_size = 1
num_samples = 50

models_to_compare = {
    "Transformer": {
        "model_name": "nicholasKluge/Aira-OPT-350M",
        "tokenizer_name": "nicholasKluge/Aira-OPT-350M",
    },
    "Mamba": {
        "model_name": "OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct",
        "tokenizer_name": "OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct",
    }
}


Using device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_config):
    tokenizer = AutoTokenizer.from_pretrained(model_config["tokenizer_name"])

    model = AutoModelForCausalLM.from_pretrained(
        model_config["model_name"],
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        trust_remote_code=True
    )

    model.to(device)
    model.eval()
    return model, tokenizer


In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=512):
    if isinstance(model, MambaLMHeadModel):
        # Specific prompt style for the Mamba-2 model
        system_prompt = "You are a helpful assistant."

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        # Typical hyperparameters for this model
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "temperature": 0.1,
            "repetition_penalty": 1.12,
            "do_sample": True,
        }
    else:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=max_input_length).to(device)

        # Typical hyperparameters for this model
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": 0.5,
            "top_p": 0.6,
            "top_k": 30,
            "repetition_penalty": 1.2,
            "pad_token_id": tokenizer.eos_token_id,
            "attention_mask": inputs["attention_mask"],
        }
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            **generation_kwargs
        )
    end_time = time.time()

    generation_time = end_time - start_time
    generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return generated_text, generation_time

In [ ]:
def rouge_evaluate_generations(references, generations):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': []
    }

    for ref, gen in zip(references, generations):
        results = scorer.score(ref, gen)
        for metric in scores.keys():
            scores[metric].append(results[metric].fmeasure)

    mean_scores = {metric: np.mean(values) for metric, values in scores.items()}
    return mean_scores

def bert_score_evaluate(references, generations, lang="en", verbose=False):
    """
    Evaluate generations using BERTScore.

    Args:
        references: List of reference texts
        generations: List of generated texts
        lang: Language of the texts (default: "en")
        verbose: Whether to print progress (default: False)

    Returns:
        Dictionary with precision, recall, and F1 scores
    """
    P, R, F1 = bert_score(generations, references, lang=lang, verbose=verbose)

    P_list = P.cpu().numpy().tolist()
    R_list = R.cpu().numpy().tolist()
    F1_list = F1.cpu().numpy().tolist()

    mean_scores = {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "F1": F1.mean().item()
    }

    individual_scores = {
        "precision": P_list,
        "recall": R_list,
        "F1": F1_list
    }

    return mean_scores, individual_scores

In [ ]:
# Load the models
model_tokenizers = {}
for model_name, model_config in models_to_compare.items():
    print(f"Loading {model_name} model...")
    model, tokenizer = load_model_and_tokenizer(model_config)

    model_tokenizers[model_name] = {"model": model, "tokenizer": tokenizer}

Loading Transformer model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct:
- configuration_mamba2attn.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_mamba2attn.py:   0%|          | 0.00/90.4k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct:
- modeling_mamba2attn.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# Load the PG-19 dataset

import itertools

print("Loading PG-19 dataset...")
num_books = 3
dataset_streamed = load_dataset("deepmind/pg19", split="validation", streaming=True)
dataset = list(itertools.islice(dataset_streamed, num_books))

STRIDE = 512 # Stride determines the number of words we skip when moving from one excerpt to another

processed_dataset = []

# Extract the excerpts
for example in tqdm(dataset):
    text = example["text"]
    input_ids = tokenizer.encode(text, truncation=False)

    for i in range(0, len(input_ids) - max_input_length - max_new_tokens, STRIDE):
        input_chunk = input_ids[i : i + max_input_length]
        target_chunk = input_ids[i + max_input_length : i + max_input_length + max_new_tokens]

        input_text = tokenizer.decode(input_chunk, skip_special_tokens=True)
        target_text = tokenizer.decode(target_chunk, skip_special_tokens=True)

        processed_dataset.append({
            "input_text": input_text,
            "target_text": target_text,
            "book_title": example["short_book_title"],
        })

input_column = "input_text"
target_column = "target_text"


Loading PG-19 dataset...


README.md:   0%|          | 0.00/8.11k [00:00<?, ?B/s]

pg19.py:   0%|          | 0.00/6.56k [00:00<?, ?B/s]

The repository for deepmind/pg19 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/deepmind/pg19.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


100%|██████████| 3/3 [00:01<00:00,  1.74it/s]


In [ ]:
import random

random.seed(477)

# Randomly sample a subset to form our eval dataset
processed_dataset = random.sample(processed_dataset, num_samples)

In [ ]:
results = {}

for model_name, model_config in models_to_compare.items():
    model = model_tokenizers[model_name]["model"]
    tokenizer = model_tokenizers[model_name]["tokenizer"]

    generations = []
    generation_times = []
    references = []
    input_lengths = []
    output_lengths = []

    for sample in tqdm(processed_dataset):
        input_text = sample[input_column]
        reference = sample[target_column]

        input_tokens = tokenizer(input_text, return_tensors="pt").input_ids.shape[1]
        input_lengths.append(input_tokens)

        generated_text, gen_time = generate_text(
            model, tokenizer, input_text, max_new_tokens=max_new_tokens
        )

        # Get output length
        output_tokens = tokenizer(generated_text, return_tensors="pt").input_ids.shape[1]
        output_lengths.append(output_tokens)

        generations.append(generated_text)
        generation_times.append(gen_time)
        references.append(reference)

    rouge_scores = rouge_evaluate_generations(
        references,
        generations
    )

    bertscore_means, bertscore_individuals = bert_score_evaluate(
        references,
        generations,
        lang="en",
        verbose=True
    )


    results[model_name] = {
        "rouge_scores": rouge_scores,
        "bertscore": bertscore_means,
        "bertscore_individuals": bertscore_individuals,
        "avg_generation_time": np.mean(generation_times),
        "generation_times": generation_times,
        "generations": generations,
        "references": references,
        "input_lengths": input_lengths,
        "output_lengths": output_lengths
    }

    del model
    torch.cuda.empty_cache()

100%|██████████| 50/50 [00:54<00:00,  1.10s/it]


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 7.74 seconds, 6.46 sentences/sec


In [ ]:
print("\n----- RESULTS -----")
for model_name, result in results.items():
    print(f"\n{model_name} Results:")
    print(f"Average Generation Time: {result['avg_generation_time']:.4f} seconds")

    print("ROUGE Scores:")
    for metric, score in result['rouge_scores'].items():
        print(f"  {metric}: {score:.4f}")

    print("BERTScore:")
    for metric, score in result['bertscore'].items():
        print(f"  {metric}: {score:.4f}")


----- RESULTS -----

Transformer Results:
Average Generation Time: 1.0898 seconds
ROUGE Scores:
  rouge1: 0.0127
  rouge2: 0.0021
  rougeL: 0.0108
BERTScore:
  precision: 0.7452
  recall: 0.6971
  F1: 0.7198

Mamba Results:
Average Generation Time: 18.8054 seconds
ROUGE Scores:
  rouge1: 0.3059
  rouge2: 0.0409
  rougeL: 0.1427
BERTScore:
  precision: 0.8247
  recall: 0.8099
  F1: 0.8172


In [ ]:
print("\n----- EXAMPLE GENERATIONS FOR LONG-RANGE DEPENDENCIES -----")

import random
num_examples = min(3, len(processed_dataset))
random_indices = random.sample(range(len(processed_dataset)), num_examples)

for i, idx in enumerate(random_indices):
    print(f"\n\n=== Example {i+1} (Dataset Index: {idx}) ===")

    # Display the input text (truncated for readability)
    input_text = processed_dataset[idx][input_column]
    print(f"\nInput text (truncated):\n{input_text[:300]}...")

    # Display the reference output
    reference = processed_dataset[idx][target_column]
    print(f"\nReference output:\n{reference[:300]}...")

    # Display generations from each model
    for model_name in models_to_compare.keys():
        generation = results[model_name]["generations"][idx]
        rouge1_score = rouge_scorer.RougeScorer(['rouge1']).score(reference, generation)["rouge1"].fmeasure
        bertscore_f1 = results[model_name]["bertscore_individuals"]["F1"][idx]

        print(f"\n{model_name} generation (ROUGE-1: {rouge1_score:.4f}, BERTScore F1: {bertscore_f1:.4f}):")
        print(f"{generation[:300]}..." if len(generation) > 300 else generation)

        # Display generation stats
        gen_time = results[model_name]["generation_times"][idx]
        input_len = results[model_name]["input_lengths"][idx]
        output_len = results[model_name]["output_lengths"][idx]

        print(f"Generation time: {gen_time:.2f}s | Input length: {input_len} tokens | "
              f"Output length: {output_len} tokens")

# Create a simple side-by-side comparison table
print("\n\n----- SIDE-BY-SIDE COMPARISON -----")

comparison_data = []
for i, idx in enumerate(random_indices):
    row = {
        "Example": i+1,
        "Input (truncated)": processed_dataset[idx][input_column] + "...",
        "Reference (truncated)": processed_dataset[idx][target_column] + "..."
    }

    for model_name in models_to_compare.keys():
        row[f"{model_name} (truncated)"] = results[model_name]["generations"][idx] + "..."
        row[f"{model_name} ROUGE-1"] = rouge_scorer.RougeScorer(['rouge1']).score(
            processed_dataset[idx][target_column],
            results[model_name]["generations"][idx]
        )["rouge1"].fmeasure

    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
pd.set_option('display.max_colwidth', 50)
print(comparison_df)
pd.reset_option('display.max_colwidth')

# Save the comparison to CSV for better viewing
comparison_df.to_csv("long_range_comparison_examples.csv", index=False)
print("\nDetailed comparison saved to long_range_comparison_examples.csv")


----- EXAMPLE GENERATIONS FOR LONG-RANGE DEPENDENCIES -----


=== Example 1 (Dataset Index: 40) ===

Input text (truncated):
city officials have regularly been nominated
at Democratic and republican conventions.

The question has arisen at the present time because of quarrels between
the mayor and aldermen, because of the petition of the city government
to the legislature to issue bonds for new waterworks above the
author...

Reference output:
draw more attention than necessary to the
arguments on the other side. Refutation of less important statements and
contentions will naturally come at the point of the argument which deals
with that part of the subject. State them fairly always, but do not
magnify their importance by dealing with the...

Transformer generation (ROUGE-1: 0.0231, BERTScore F1: 0.7572):
 thus said thus x yet came thus
baptious
n so
 Thus a& therefore this</s> this Therefore the reason so this year thus thus...

 Hence this period) therefore this article...

Generati